In [2]:
import numpy as np

T = np.arange(24).reshape(2, 3, 4)
T

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [3]:
passed = True

for i in range(2):
    for j in range(3):
        for k in range(4):
            if T[i, j, k] != T.reshape(-1)[12*i + 4*j + k]:
                passed = False

passed

True

In [4]:
U1 = T.reshape(2, 12)
U1

array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]])

In [9]:
rank(U1)

2

In [5]:
U2 = T.transpose(1, 0, 2).reshape(3, 8)
U2

array([[ 0,  1,  2,  3, 12, 13, 14, 15],
       [ 4,  5,  6,  7, 16, 17, 18, 19],
       [ 8,  9, 10, 11, 20, 21, 22, 23]])

In [10]:
rank(U2)

2

In [7]:
def rref(A, tol=1e-10):
    """Return (R, pivots): the RREF of A and the list of pivot columns."""
    R = np.array(A, dtype=float)
    m, n = R.shape #m is number of rows  and n is number of columns

    pivots = []
    r = 0 #to keep track of the row where the next pivot is places

    for c in range(n):
        if r == m: #if all rows contain pivots
            break

        pivot_row = r 

        for i in range(r + 1, m): #to find the entry with the largest absolute value
            if abs(R[i, c]) > abs(R[pivot_row, c]):
                pivot_row = i
        
        #if it is smaller than tol,the column has no pivot, so we skip it
        if abs(R[pivot_row, c]) < tol:
            continue
    
        #otherwise, swaps its row with row r
        R[[r, pivot_row]] = R[[pivot_row, r]]
        
        #divide row r by the pivot
        pivot = R[r, c]
        R[r] = R[r] / pivot
        
        #subtracts multiples of row r from all the other rows
        for i in range(m):
            if i != r:
                factor = R[i, c]
                R[i] = R[i] - factor * R[r]

        pivots.append(c)
        r += 1
    
    #replaces very small values with zero
    R[abs(R) < tol] = 0

    return R, pivots

In [8]:
def rank(A):
    R, pivots = rref(A)
    rank = len(pivots)
    return rank

In [11]:
rng = np.random.default_rng(0)

A = rng.normal(size=(4, 4))
B = rng.normal(size=(4, 4))
x = rng.normal(size=4)
y = rng.normal(size=4)

In [15]:
#matrix product
product = np.einsum("ij,jk->ik", A, B)
np.allclose(product, A @ B)

True

In [16]:
#trace
trace = np.einsum("ii->", A)
np.allclose(trace, np.trace(A))

True

In [18]:
#dot product
dot = np.einsum("i,i->", x, y)
np.allclose(dot, x @ y)

True

In [19]:
#outer product
outer = np.einsum("i,j->ij", x, y)
np.allclose(outer, np.outer(x, y))

True

In [20]:
#transpose
transpose = np.einsum("ij->ji", A)
np.allclose(transpose, A.T)

True

In [22]:
#bilinear
bilinear = np.einsum("i,ij,j->", x, A, y)
np.allclose(bilinear, x @ A @ y)

True

In [24]:
#batch matrix multiplication
A_batch = rng.normal(size=(100, 4, 4))
B_batch = rng.normal(size=(100, 4, 4))

batch = np.einsum("bij,bjk->bik", A_batch, B_batch)
np.allclose(batch, A_batch @ B_batch)

True

In [25]:
#100 random 4x4 matrices
A_batch = rng.normal(size=(100, 4, 4))
B_batch = rng.normal(size=(100, 4, 4))

traces = np.einsum("bii->b", A_batch)
products = np.einsum("bij,bjk->bik", A_batch, B_batch)

In [27]:
traces

array([ 5.1464565 , -3.49382061, -0.6594576 , -1.76246382,  1.2429961 ,
       -0.86132774, -2.71087507, -1.16878113, -1.83766266,  0.75144009,
        3.15315535,  1.85291089,  0.07596216,  3.50816741, -1.60114862,
        1.2055819 ,  1.69570158,  2.06404745, -2.27524419,  1.32691997,
       -2.33876705,  0.24323122,  1.42246663,  1.68764774, -0.328066  ,
       -0.52633338, -1.35335482,  1.27540657, -1.33764077, -0.83832731,
        0.25442559,  0.61492534, -0.98088074,  0.18873543, -2.61169994,
        3.54039787,  1.51975405,  0.15463798,  1.53605785, -4.21343863,
       -0.50079318, -1.30068345,  1.16867438,  0.14863082, -2.08556702,
       -0.53530902,  2.27320867,  0.4306458 ,  0.74967499,  0.10677942,
       -0.99456473, -1.60991918, -0.90892471,  0.56976117, -5.67531274,
       -0.39397819,  2.35003105, -0.43075385, -0.78798714,  0.87494063,
       -1.80604433, -0.65679892, -0.25813131, -0.00704902,  0.58400892,
       -2.84343931, -3.09090992,  0.10609349,  0.17135803, -2.35

In [26]:
products

array([[[-5.08119875e-01,  2.34656192e+00, -2.78011073e-01,
         -8.64964805e-01],
        [ 7.52099245e-01,  3.10976657e-01,  1.58465104e+00,
         -4.31017722e-01],
        [-1.12286983e+00,  1.43902102e+00,  1.03121428e+00,
         -1.47508978e+00],
        [-7.39046097e-01,  2.28172054e+00, -9.88221252e-01,
         -7.27470935e-01]],

       [[ 3.49385087e+00,  4.05374653e+00,  2.19886545e+00,
         -4.57910062e-01],
        [ 1.55144740e+00,  1.53494935e+00,  5.28175601e-01,
          7.50776300e-01],
        [ 4.76590383e+00,  6.56731646e+00,  3.78523860e+00,
         -1.97568145e+00],
        [ 1.06602944e+00,  5.37304764e-01, -1.14241448e-01,
         -1.51264464e+00]],

       [[-7.04120133e-01, -3.03965064e-02, -2.23818132e-01,
         -1.69493085e-01],
        [ 9.62102849e-01, -7.67727318e-01, -1.46540165e+00,
          2.99669392e+00],
        [-4.10949731e+00, -1.54937327e+00,  9.76621265e-01,
         -3.12182567e+00],
        [ 2.02150275e+00,  4.83883542e-

In [28]:
traces_loop = np.zeros(100)

for b in range(100):
    traces_loop[b] = np.trace(A_batch[b])
traces_loop

array([ 5.1464565 , -3.49382061, -0.6594576 , -1.76246382,  1.2429961 ,
       -0.86132774, -2.71087507, -1.16878113, -1.83766266,  0.75144009,
        3.15315535,  1.85291089,  0.07596216,  3.50816741, -1.60114862,
        1.2055819 ,  1.69570158,  2.06404745, -2.27524419,  1.32691997,
       -2.33876705,  0.24323122,  1.42246663,  1.68764774, -0.328066  ,
       -0.52633338, -1.35335482,  1.27540657, -1.33764077, -0.83832731,
        0.25442559,  0.61492534, -0.98088074,  0.18873543, -2.61169994,
        3.54039787,  1.51975405,  0.15463798,  1.53605785, -4.21343863,
       -0.50079318, -1.30068345,  1.16867438,  0.14863082, -2.08556702,
       -0.53530902,  2.27320867,  0.4306458 ,  0.74967499,  0.10677942,
       -0.99456473, -1.60991918, -0.90892471,  0.56976117, -5.67531274,
       -0.39397819,  2.35003105, -0.43075385, -0.78798714,  0.87494063,
       -1.80604433, -0.65679892, -0.25813131, -0.00704902,  0.58400892,
       -2.84343931, -3.09090992,  0.10609349,  0.17135803, -2.35

In [29]:
np.allclose(traces, traces_loop)

True

In [31]:
products_loop = np.zeros((100, 4, 4))

for b in range(100):
    products_loop[b] = A_batch[b] @ B_batch[b]

np.allclose(products, products_loop)

True

matches, so batch operations work correctly

In [32]:
A = rng.normal(size=(3, 4))
B = rng.normal(size=(2, 5))
X = rng.normal(size=(4, 5))

In [33]:
left = (A @ X @ B.T).reshape(-1)
left

array([ 1.694662  ,  2.81411157, -0.96066795,  3.93310495, -7.87008981,
       -2.44523519])

In [34]:
right = np.kron(A, B) @ X.reshape(-1)
right

array([ 1.694662  ,  2.81411157, -0.96066795,  3.93310495, -7.87008981,
       -2.44523519])

In [35]:
np.allclose(left, right)

True

they equal

In [36]:
n = 100
rows = n * n
cols = n * n
elements = rows * cols
elements

100000000

In [37]:
memory_mb = elements * 8 / 1e6
memory_mb

800.0

Computing A @ X @ B.T directly is much cheaper because it avoids creating the large Kronecker product and uses two smaller matrix multiplications instead.

In [38]:
rng = np.random.default_rng(0)

def rank_one(u, v, w):
    return np.einsum("i,j,k->ijk", u, v, w)

In [39]:
u = rng.normal(size=3)
v = rng.normal(size=4)
w = rng.normal(size=2)
T = rank_one(u, v, w)
T

array([[[ 0.01249116, -0.00928164],
        [-0.06378574,  0.04739645],
        [ 0.04305755, -0.03199421],
        [ 0.15527602, -0.11537895]],

       [[-0.01312447,  0.00975223],
        [ 0.06701974, -0.04979949],
        [-0.0452406 ,  0.03361635],
        [-0.16314866,  0.12122877]],

       [[ 0.06362529, -0.04727722],
        [-0.32490067,  0.24141975],
        [ 0.21931901, -0.16296655],
        [ 0.79091789, -0.58769715]]])

In [40]:
M = np.outer(u, v)
M

array([[ 0.01318911, -0.06734983,  0.04546343,  0.16395221],
       [-0.01385782,  0.07076453, -0.04776847, -0.17226475],
       [ 0.06718041, -0.3430548 ,  0.23157366,  0.83511117]])

In [41]:
passed = True

for k in range(len(w)):
    if not np.allclose(T[:, :, k], w[k] * M):
        passed = False

passed

True

All slices are multiples of u v.T